# 2. Базовые и контентные модели

В этом ноутбуке:
- делим оценки на train и test (это разбиение используется во всех следующих ноутбуках)
- строим неперсонализированную модель, то есть топ книг по среднему рейтингу с порогом на число оценок
- строим контентную модель на TF-IDF по названию и тегам книги

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

from utils import (
    ARTIFACTS_DIR, DATA_DIR, N_BOOKS, TOP_K,
    get_eval_users, load_books, load_ratings, recommend_top_k, save_recs, split_ratings, to_sparse,
)

ARTIFACTS_DIR.mkdir(exist_ok=True)

## Разбиение на train и test

Временных меток в датасете нет, поэтому разбиение случайное. Делим внутри каждого пользователя:
20% его оценок уходит в тест, остальное в обучение. Так у каждого тестового пользователя остаётся история в train,
то есть оцениваем рекомендации для пользователей с историей.
У пользователей с одной-двумя оценками в тест ничего не попадает.

In [2]:
ratings = load_ratings()
books = load_books()
train, test = split_ratings(ratings)

eval_users = get_eval_users(test)
print(f"train: {len(train)} оценок, test: {len(test)} оценок ({len(test) / len(ratings):.1%})")
print(f"Пользователей в train: {train['user_id'].nunique()}, в test: {test['user_id'].nunique()}")
print(f"Пользователей с релевантными книгами в тесте (на них считаем метрики): {len(eval_users)}")

train: 784517 оценок, test: 194961 оценок (19.9%)
Пользователей в train: 53424, в test: 45071
Пользователей с релевантными книгами в тесте (на них считаем метрики): 37968


In [3]:
train_matrix = to_sparse(train)
train_matrix

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 784517 stored elements and shape (53424, 10000)>

## Неперсонализированная модель: популярные книги

Сортируем книги по средней оценке, но учитываем только те, у которых набралось достаточно оценок.
Без порога наверху окажутся книги с парой случайных пятёрок.

In [4]:
def get_popular_books(ratings: pd.DataFrame, n: int = 10, min_ratings: int = 50) -> pd.DataFrame:
    """Top-n книг по средней оценке среди книг, у которых не меньше min_ratings оценок."""
    stats = ratings.groupby("book_id")["rating"].agg(mean_rating="mean", n_ratings="count")
    stats = stats[stats["n_ratings"] >= min_ratings]
    return stats.sort_values(["mean_rating", "n_ratings"], ascending=False).head(n).reset_index()


train.groupby("book_id").size().describe().round(1)

count    10000.0
mean        78.5
std          5.9
min          6.0
25%         76.0
50%         79.0
75%         82.0
max         95.0
dtype: float64

В train у книги в среднем 78 оценок (максимум 95), поэтому порог берём 50. Он отсекает книги с малым числом оценок
и оставляет большую часть каталога.

In [5]:
MIN_RATINGS = 50

popular = get_popular_books(train, n=N_BOOKS, min_ratings=MIN_RATINGS)
print(f"Книг, прошедших порог: {len(popular)} из {N_BOOKS}")

popular.head(10).merge(books[["book_id", "title", "authors"]], on="book_id")[
    ["book_id", "title", "authors", "mean_rating", "n_ratings"]
]

Книг, прошедших порог: 9960 из 10000


,book_id,title,authors,mean_rating,n_ratings
0,7947,ESV Study Bible,"Anonymous, Lane T. Dennis, Wayne A. Grudem",4.814286,70
1,6590,The Authoritative Calvin and Hobbes: A Calvin ...,Bill Watterson,4.807692,78
2,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.787500,80
3,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.773333,75
4,4483,It's a Magical World: A Calvin and Hobbes Coll...,Bill Watterson,4.771084,83
5,3275,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré",4.768293,82
6,9566,Attack of the Deranged Mutant Killer Monster S...,Bill Watterson,4.766234,77
7,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,4.740741,81
8,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,4.738095,84
9,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,4.736842,76


В топе сборники комиксов "Кельвин и Хоббс", Библия с комментариями, бокс-сет "Гарри Поттера". Это книги с узкой,
но очень лояльной аудиторией. Так обычно и бывает при сортировке по средней оценке: наверх попадает то,
что высоко оценили те, кто это выбрал.

Чтобы потом сравнить модель с остальными, получим для каждого тестового пользователя top-10 таких книг,
исключая уже оценённые. Оценка релевантности книги одинакова для всех пользователей, это её средний рейтинг.

In [6]:
popularity_scores = np.zeros(N_BOOKS, dtype=np.float32)
popularity_scores[popular["book_id"] - 1] = popular["mean_rating"]

recs_popular = recommend_top_k(
    lambda user_idx: np.tile(popularity_scores, (len(user_idx), 1)),
    seen=train_matrix, user_ids=eval_users, k=TOP_K,
)
save_recs("popular", eval_users, recs_popular)
popular.to_csv(ARTIFACTS_DIR / "popular.csv", index=False)
recs_popular[:3]

array([[7947, 6590, 6920, 5207, 4483, 3275, 9566, 5580, 6361, 8946],
       [7947, 6590, 6920, 5207, 4483, 3275, 9566, 5580, 6361, 8946],
       [7947, 6590, 6920, 5207, 4483, 3275, 9566, 5580, 6361, 8946]],
      dtype=int32)

## Контентная модель

### Текстовый профиль книги

В профиль входит название книги (`original_title`, а если его нет, то `title`) и все её теги через пробел.

In [7]:
tags = pd.read_csv(DATA_DIR / "tags.csv")
book_tags = pd.read_csv(DATA_DIR / "book_tags.csv")
book_tags = book_tags[book_tags["count"] > 0].drop_duplicates(["goodreads_book_id", "tag_id"])

tags_per_book = (
    book_tags.merge(tags, on="tag_id")
    .groupby("goodreads_book_id")["tag_name"]
    .agg(" ".join)
)

books["profile"] = (
    books["original_title"].fillna(books["title"])
    + " "
    + books["goodreads_book_id"].map(tags_per_book).fillna("")
)
print(books.loc[0, "profile"][:300], "...")

The Hunger Games favorites currently-reading young-adult fiction dystopian to-read dystopia fantasy ya science-fiction books-i-own sci-fi series owned favourites romance adventure hunger-games book-club kindle teen read-in-2012 post-apocalyptic my-books the-hunger-games favorite-books action suzanne ...


### TF-IDF

Теги вида `science-fiction` или `young-adult` хочется оставить целыми токенами, поэтому в `token_pattern` разрешаем дефис.
Стоп-слова убираем (они есть в названиях). Токены, которые встретились только в одной книге, отбрасываем через `min_df=2`,
для поиска похожих книг они бесполезны.

Служебные теги (`to-read`, `favorites`, `owned`) отдельно не чистим. Они стоят почти у всех книг,
и IDF сам даёт им вес, близкий к минимальному.

In [8]:
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b[\w-]+\b", stop_words="english", min_df=2)
tfidf = vectorizer.fit_transform(books["profile"])

print(f"Матрица TF-IDF: {tfidf.shape}, тип: {type(tfidf).__name__}")
print(f"Ненулевых элементов: {tfidf.nnz} ({tfidf.nnz / np.prod(tfidf.shape):.2%})")

idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())
print("\nIDF некоторых тегов:")
print(idf[["to-read", "favorites", "fiction", "fantasy", "dystopia", "vampires"]].round(2).to_string())

Матрица TF-IDF: (10000, 19268), тип: csr_matrix
Ненулевых элементов: 994563 (0.52%)

IDF некоторых тегов:
to-read      1.00
favorites    1.01
fiction      1.09
fantasy      1.85
dystopia     3.63
vampires     3.76


Матрица хранится в разреженном формате (CSR), заполнено около половины процента ячеек.
По IDF видно, что `to-read` и `favorites` получили минимальный вес, а у содержательных тегов вес заметно больше.

### Поиск похожих книг

`TfidfVectorizer` нормирует строки по L2, поэтому косинусная близость двух книг равна скалярному произведению
их векторов. Считаем близость одной книги со всеми остальными, полную матрицу 10 000 x 10 000 не строим.

In [9]:
def get_similar_books(book_id: int, N: int = 5) -> pd.DataFrame:
    """N книг, наиболее близких к book_id по косинусной мере между TF-IDF-векторами."""
    idx = book_id - 1
    sims = linear_kernel(tfidf[idx], tfidf).ravel()
    sims[idx] = -1  # саму книгу не возвращаем
    top = np.argsort(-sims)[:N]
    result = books.loc[top, ["book_id", "title", "authors"]].copy()
    result["similarity"] = sims[top].round(3)
    return result.reset_index(drop=True)


print(books.loc[0, "title"])
get_similar_books(1)

The Hunger Games (The Hunger Games, #1)


,book_id,title,authors,similarity
0,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.824
1,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.807
2,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.748
3,12,"Divergent (Divergent, #1)",Veronica Roth,0.581
4,69,"Insurgent (Divergent, #2)",Veronica Roth,0.516


In [10]:
book_id = int(books.loc[books["title"] == "To Kill a Mockingbird", "book_id"].iloc[0])
print(books.loc[book_id - 1, "title"])
get_similar_books(book_id)

To Kill a Mockingbird


,book_id,title,authors,similarity
0,8,The Catcher in the Rye,J.D. Salinger,0.636
1,5,The Great Gatsby,F. Scott Fitzgerald,0.624
2,32,Of Mice and Men,John Steinbeck,0.567
3,28,Lord of the Flies,William Golding,0.532
4,58,The Adventures of Huckleberry Finn,"Mark Twain, John Seelye, Guy Cardwell",0.483


In [11]:
book_id = int(books.loc[books["title"].str.startswith("A Brief History of Time"), "book_id"].iloc[0])
print(books.loc[book_id - 1, "title"])
get_similar_books(book_id, N=7)

A Brief History of Time


,book_id,title,authors,similarity
0,1235,Cosmos,Carl Sagan,0.596
1,3609,"The Fabric of the Cosmos: Space, Time, and the...",Brian Greene,0.586
2,8400,Black Holes and Baby Universes,Stephen Hawking,0.568
3,5256,Death by Black Hole - And Other Cosmic Quandaries,Neil deGrasse Tyson,0.563
4,1224,"The Elegant Universe: Superstrings, Hidden Dim...",Brian Greene,0.563
5,3696,The Universe in a Nutshell,Stephen Hawking,0.562
6,1985,The Grand Design,"Stephen Hawking, Leonard Mlodinow",0.560


Результаты осмысленные. Для "Голодных игр" модель находит продолжения серии и "Дивергент" (тоже подростковая антиутопия),
для "Убить пересмешника" находит американскую классику из школьной программы, для Хокинга научно-популярные книги по физике.

Ограничение тоже видно. Модель опирается только на теги и слова названия. Она хорошо находит книги той же серии
и того же жанра, но не знает, хорошая книга или плохая, и ничего не знает о вкусах конкретного человека.

### Контентные рекомендации для пользователя

Функция `get_similar_books` работает от книги. Чтобы рекомендовать пользователю, соберём его профиль как сумму
TF-IDF-векторов оценённых им книг с весами-оценками и посчитаем близость профиля ко всем книгам.
В матричном виде это два умножения разреженных матриц: `(R @ T) @ T.T`. Эти рекомендации пригодятся в сравнении моделей
и в гибриде.

In [12]:
tfidf_t = tfidf.T.tocsr()


def content_scores(user_idx):
    user_profiles = train_matrix[user_idx] @ tfidf
    return (user_profiles @ tfidf_t).toarray()


recs_content = recommend_top_k(content_scores, seen=train_matrix, user_ids=eval_users, k=TOP_K)
save_recs("content", eval_users, recs_content)
sp.save_npz(ARTIFACTS_DIR / "tfidf.npz", tfidf)

In [13]:
user_id = int(eval_users[100])
history = train[train["user_id"] == user_id].merge(books[["book_id", "title"]], on="book_id")
print(f"Пользователь {user_id}, оценённые книги:")
print(history.sort_values("rating", ascending=False)[["rating", "title"]].to_string(index=False))

print("\nКонтентные рекомендации:")
print("\n".join(books.set_index("book_id").loc[recs_content[100, :5], "title"]))

Пользователь 157, оценённые книги:
 rating                           title
      5      Citizen: An American Lyric
      5                            HHhH
      4 Concerning the Spiritual in Art

Контентные рекомендации:
Between the World and Me
The Fire Next Time
The Souls of Black Folk
for colored girls who have considered suicide/when the rainbow is enuf
Kindred


## Итог

- Разбиение на train/test зафиксировано (`utils.split_ratings`, seed 42) и одинаково во всех ноутбуках.
- Неперсонализированная модель: топ по средней оценке среди книг с не менее чем 50 оценками.
- Контентная модель: TF-IDF по названию и тегам, `get_similar_books(book_id, N)` по косинусной близости.
- Top-10 рекомендаций обеих моделей для тестовых пользователей сохранены в `artifacts/`, метрики считаются в ноутбуке 5.